# Visual-Doc Assistant — Evaluation 2: OCR/Text RAG vs. ColPali (Visual RAG)

**Question answered:** *When only the retrieval method changes, does page-image retrieval (ColPali + MaxSim) find the right page more often than OCR + text retrieval?*

**What is held fixed (loaded from the frozen `benchmark_v1`)**
- the 30 questions, gold pages and the 521-page index (same page PNGs)
- the Gemini generator, system prompt and 1 page per question
- the metric code (`eval_metrics.py`, imported unchanged)

**What varies:** the retrieval method only.

**Arms**
| Arm | Method |
|---|---|
| `colpali` | ColPali + exhaustive MaxSim (ranks loaded from Evaluation 1, nothing recomputed) |
| `ocr-bm25` | Tesseract OCR text, BM25 over whole pages |
| `ocr-dense` | Tesseract OCR text, BGE-small embeddings over overlapping chunks, page score = best chunk |
| `ocr-hybrid` | Reciprocal-rank fusion of the two above |
| `native-*` (optional) | Same three, but over the PDF's own text layer (no OCR errors). Needs the PDF files. |

**Fairness controls (do not remove them):** the text arms are given strong, standard baselines (stop-word removal, chunking, hybrid fusion), the question text is passed unchanged to every arm, the best text arm for the generation step is chosen by *retrieval* MRR before any generation result is seen, and the results are reported per modality with a paired significance test, since only 30 questions are available.

## 1. Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pytesseract rank_bm25 sentence-transformers pymupdf scipy chromadb google-genai pandas tabulate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [5]:
import os, sys, json, time, math, re, hashlib, platform, base64, random, importlib
import numpy as np, pandas as pd
import importlib.metadata as _im
from PIL import Image

BASE     = "/content/drive/MyDrive/Visual-Doc"
EVAL_DIR = f"{BASE}/eval_results"
FROZEN   = f"{EVAL_DIR}/benchmark_v1"
E2       = f"{EVAL_DIR}/eval2"
DB_PATH  = f"{BASE}/chroma_db_storage"
os.makedirs(E2, exist_ok=True)

if BASE not in sys.path: sys.path.append(BASE)
import eval_metrics; importlib.reload(eval_metrics)
from eval_metrics import (metrics_from_ranks, metrics_table, rank_of_gold, hit_at_k,
                          reciprocal_rank, is_abstention, strict_keyword_score)
print("Setup OK")

Setup OK


## 2. Load the frozen benchmark and the ColPali arm's ranks

Nothing here is recomputed. The hash check confirms the questions are exactly the ones Evaluation 1 used.

In [6]:
with open(f"{FROZEN}/qa_set.json") as f: _q = json.load(f)
QA_SET, QA_HASH = _q["questions"], _q["qa_hash"]
with open(f"{FROZEN}/eval_config.json") as f: EVAL_CFG = json.load(f)
with open(f"{FROZEN}/visual_arm_ranks.json") as f: VIS = json.load(f)
with open(f"{EVAL_DIR}/eval1_summary_final.json") as f: E1_SUM = json.load(f)
E1_CSV = pd.read_csv(f"{EVAL_DIR}/eval1_final_per_question.csv")

recomputed = hashlib.sha256(json.dumps(QA_SET, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:16]
assert recomputed == QA_HASH == VIS["qa_hash"] == EVAL_CFG["qa_hash"], "Benchmark hash mismatch -- do not continue"
vis_by_idx = {r["idx"]: r for r in VIS["results"]}
print(f"Benchmark OK: {len(QA_SET)} questions, hash {QA_HASH}")
print("ColPali arm:", VIS["arm"])

Benchmark OK: 30 questions, hash 6057af19ccde7e1c
ColPali arm: ColPali + exhaustive MaxSim


## 3. The page list (same 521 pages, same PNGs as Evaluation 1)

In [7]:
import chromadb
col = chromadb.PersistentClient(path=DB_PATH).get_or_create_collection(name="visual_doc_collection", metadata={"hnsw:space": "cosine"})
metas = col.get(include=["metadatas"])["metadatas"]
PAGE_IMG = {(m["doc_name"], m["page_number"]): m["image_path"] for m in metas}
PAGES = sorted(PAGE_IMG)                      # fixed order used by every arm
PAGE_IDX = {k: i for i, k in enumerate(PAGES)}
print(len(PAGES), "pages;", pd.Series([d for d, _ in PAGES]).value_counts().to_dict())

521 pages; {'WorldDevelopmentReport': 359, 'AIFoundations': 136, '2407.01449v6': 26}


## 4. OCR every page (Tesseract)

Resumable: results are cached on Drive every 25 pages, so a disconnect only costs the unfinished pages. Expect roughly 20-40 minutes on a Colab CPU runtime. The per-page OCR time is recorded for the efficiency comparison.

In [8]:
import pytesseract
from concurrent.futures import ThreadPoolExecutor

OCR_CACHE = f"{E2}/ocr_cache.json"
ocr = json.load(open(OCR_CACHE)) if os.path.exists(OCR_CACHE) else {}
k2s = lambda k: f"{k[0]}|{k[1]}"

def _ocr_one(key):
    t0 = time.perf_counter()
    text = pytesseract.image_to_string(Image.open(PAGE_IMG[key]).convert("RGB"), lang="eng", config="--psm 3")
    return key, text, time.perf_counter() - t0

todo = [k for k in PAGES if k2s(k) not in ocr]
print(f"{len(ocr)} cached, {len(todo)} to OCR")
t_wall0 = time.time()
with ThreadPoolExecutor(max_workers=max(1, os.cpu_count() or 2)) as ex:
    for n, (key, text, sec) in enumerate(ex.map(_ocr_one, todo), 1):
        ocr[k2s(key)] = {"text": text, "sec": sec}
        if n % 25 == 0 or n == len(todo):
            json.dump(ocr, open(OCR_CACHE, "w"))
            print(f"  {n}/{len(todo)}  ({time.time()-t_wall0:.0f}s elapsed)")
json.dump(ocr, open(OCR_CACHE, "w"))

OCR_TEXT = {k: ocr[k2s(k)]["text"] for k in PAGES}
ocr_secs = [ocr[k2s(k)]["sec"] for k in PAGES]
words = [len(t.split()) for t in OCR_TEXT.values()]
print(f"\nOCR done. mean {np.mean(ocr_secs):.2f} s/page per call; words/page mean {np.mean(words):.0f}, "
      f"pages with <20 words: {sum(w < 20 for w in words)}")

521 cached, 0 to OCR

OCR done. mean 20.08 s/page per call; words/page mean 510, pages with <20 words: 29


## 4b. (Optional) the PDF's own text layer

OCR errors are a weakness of the OCR arm that the PDF's embedded text does not have. If you have the three PDFs, fill `PDF_PATHS` and re-run this cell; it adds a `native-*` set of arms so the visual arm is also compared against *clean* text. This is the strictest text baseline. Leave it empty to skip.

The page numbers must match the ones used in the benchmark (the PDF page index, starting at 1).

In [9]:
PDF_PATHS = {
    # "2407.01449v6":           "/content/drive/MyDrive/Visual-Doc/pdfs/2407.01449v6.pdf",
    # "WorldDevelopmentReport": "/content/drive/MyDrive/Visual-Doc/pdfs/WorldDevelopmentReport.pdf",
    # "AIFoundations":          "/content/drive/MyDrive/Visual-Doc/pdfs/AIFoundations.pdf",
}
TEXT_SOURCES = {"ocr": OCR_TEXT}
if PDF_PATHS:
    import fitz
    NATIVE = {}
    for doc, path in PDF_PATHS.items():
        with fitz.open(path) as d:
            for i, page in enumerate(d, 1): NATIVE[(doc, i)] = page.get_text()
    missing = [k for k in PAGES if k not in NATIVE]
    assert not missing, f"{len(missing)} indexed pages have no native text, e.g. {missing[:3]}"
    TEXT_SOURCES["native"] = {k: NATIVE[k] for k in PAGES}
print("Text sources:", list(TEXT_SOURCES))

Text sources: ['ocr']


## 5. Build the text retrievers

- **BM25** over whole pages (Okapi BM25, English stop-words removed).
- **Dense**: `BAAI/bge-small-en-v1.5`, each page split into 180-word chunks with 60-word overlap, page score = best chunk (so long pages are not truncated).
- **Hybrid**: reciprocal-rank fusion (k = 60) of BM25 and dense.

The question string is used exactly as written, the same input the ColPali arm received.

In [10]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
BGE_PREFIX = "Represent this sentence for searching relevant passages: "

def tok(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in ENGLISH_STOP_WORDS]

def chunk_words(text, size=180, stride=120):
    w = text.split()
    if not w: return [" "]
    out, i = [], 0
    while True:
        out.append(" ".join(w[i:i + size]))
        if i + size >= len(w): break
        i += stride
    return out

def build_dense(texts, tag):
    chunks, owner = [], []
    for pi, key in enumerate(PAGES):
        for c in chunk_words(texts[key]): chunks.append(c); owner.append(pi)
    cache = f"{E2}/dense_{tag}.npy"
    if os.path.exists(cache) and np.load(cache).shape[0] == len(chunks):
        emb = np.load(cache)
    else:
        emb = embedder.encode(chunks, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
        np.save(cache, emb)
    return emb, np.array(owner)

def dense_scores(q, emb, owner):
    qe = embedder.encode([BGE_PREFIX + q], normalize_embeddings=True)[0]
    s = emb @ qe
    page = np.full(len(PAGES), -1e9); np.maximum.at(page, owner, s)
    return page

def rank_positions(scores):
    order = np.argsort(-scores, kind="stable")
    pos = np.empty(len(scores), dtype=int); pos[order] = np.arange(len(scores))
    return pos

SCORES, LAT = {}, {}     # SCORES[arm] -> (n_questions, n_pages); LAT[arm] -> list of ms per query
ARMS = []
for tag, texts in TEXT_SOURCES.items():
    bm25 = BM25Okapi([tok(texts[k]) for k in PAGES])
    emb, owner = build_dense(texts, tag)
    b, d, h, tb, td = [], [], [], [], []
    for qa in QA_SET:
        t0 = time.perf_counter(); sb = bm25.get_scores(tok(qa["question"])); tb.append((time.perf_counter() - t0) * 1000)
        t0 = time.perf_counter(); sd = dense_scores(qa["question"], emb, owner); td.append((time.perf_counter() - t0) * 1000)
        rrf = 1.0 / (60 + rank_positions(sb) + 1) + 1.0 / (60 + rank_positions(sd) + 1)
        b.append(sb); d.append(sd); h.append(rrf)
    for name, arr, lat in ((f"{tag}-bm25", b, tb), (f"{tag}-dense", d, td), (f"{tag}-hybrid", h, [x + y for x, y in zip(tb, td)])):
        SCORES[name] = np.array(arr); LAT[name] = lat; ARMS.append(name)
print("Arms:", ["colpali"] + ARMS)

Arms: ['colpali', 'ocr-bm25', 'ocr-dense', 'ocr-hybrid']


## 6. Retrieval comparison (same metric code as Evaluation 1)

In [11]:
res = pd.DataFrame([{"idx": i, "doc_name": q["doc_name"], "gold_page": q["page"], "modality": q["modality"],
                     "question": q["question"]} for i, q in enumerate(QA_SET)])
gold = [(q["doc_name"], q["page"]) for q in QA_SET]
res["colpali"] = [vis_by_idx[i]["rank_of_gold"] for i in range(len(QA_SET))]
TOP10 = {"colpali": [[tuple(x) for x in vis_by_idx[i]["top10"]] for i in range(len(QA_SET))]}
for arm in ARMS:
    tops = []
    for i in range(len(QA_SET)):
        order = np.argsort(-SCORES[arm][i], kind="stable")[:10]
        tops.append([PAGES[j] for j in order])
    TOP10[arm] = tops
    res[arm] = [rank_of_gold(tops[i], gold[i]) for i in range(len(QA_SET))]

ALL = ["colpali"] + ARMS
overall = pd.DataFrame({a: metrics_from_ranks(res[a].tolist()) for a in ALL}).T.drop(columns="n")
print("=== Overall (30 questions, 521 pages) ===")
display(overall.round(3))
print("\nColPali row should equal Evaluation 1:", abs(overall.loc["colpali", "Recall@1"] - E1_SUM["retrieval"]["overall"]["Recall@1"]) < 1e-9)

print("\n=== Recall@1 by modality ===")
by_mod = pd.DataFrame({a: metrics_table(res, "modality", rank_col=a)["Recall@1"] for a in ALL})
by_mod.insert(0, "n", res.groupby("modality").size())
display(by_mod.round(3))

print("=== MRR@10 by document ===")
display(pd.DataFrame({a: metrics_table(res, "doc_name", rank_col=a)["MRR@10"] for a in ALL}).round(3))

=== Overall (30 questions, 521 pages) ===


,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@5
colpali,0.867,0.967,0.967,1.000,0.910,0.921
ocr-bm25,0.767,0.900,0.933,0.967,0.836,0.856
ocr-dense,0.367,0.733,0.933,0.967,0.576,0.661
ocr-hybrid,0.467,0.900,0.933,1.000,0.697,0.753



ColPali row should equal Evaluation 1: True

=== Recall@1 by modality ===


,n,colpali,ocr-bm25,ocr-dense,ocr-hybrid
chart,15,0.867,0.800,0.467,0.533
diagram,2,1.000,0.500,0.000,0.000
infographic,1,1.000,1.000,0.000,0.000
map,2,1.000,1.000,0.500,0.500
table,7,0.857,0.571,0.286,0.429
text,3,0.667,1.000,0.333,0.667


=== MRR@10 by document ===


,colpali,ocr-bm25,ocr-dense,ocr-hybrid
2407.01449v6,0.781,0.725,0.537,0.631
AIFoundations,0.950,0.833,0.508,0.710
WorldDevelopmentReport,1.000,0.950,0.683,0.750


## 7. Is the difference real? Paired comparison on the same 30 questions

With 30 questions a gap can be noise, so for each text arm this reports wins/losses, an exact sign test on the questions where exactly one arm got rank 1, and a bootstrap 95% interval for the difference (ColPali minus the text arm).

In [12]:
from scipy.stats import binomtest
rng = np.random.default_rng(0)

def paired(arm):
    a1 = np.array([hit_at_k(r, 1) for r in res["colpali"]]); b1 = np.array([hit_at_k(r, 1) for r in res[arm]])
    ma = np.array([reciprocal_rank(r) for r in res["colpali"]]); mb = np.array([reciprocal_rank(r) for r in res[arm]])
    w, l = int((a1 & ~b1).sum()), int((~a1 & b1).sum())
    p = binomtest(w, w + l, 0.5).pvalue if (w + l) else 1.0
    n = len(a1); boots_r, boots_m = [], []
    for _ in range(10000):
        ix = rng.integers(0, n, n)
        boots_r.append(a1[ix].mean() - b1[ix].mean()); boots_m.append(ma[ix].mean() - mb[ix].mean())
    return {"ColPali wins @1": w, "text wins @1": l, "ties": n - w - l, "sign-test p": round(p, 4),
            "dRecall@1": round(a1.mean() - b1.mean(), 3), "dRecall@1 95% CI": tuple(np.round(np.percentile(boots_r, [2.5, 97.5]), 3)),
            "dMRR@10": round(ma.mean() - mb.mean(), 3), "dMRR@10 95% CI": tuple(np.round(np.percentile(boots_m, [2.5, 97.5]), 3))}

PAIRED = pd.DataFrame({arm: paired(arm) for arm in ARMS}).T
display(PAIRED)
print("A confidence interval that excludes 0 means the gap is unlikely to be sampling noise.")

print("\n=== Questions where the arms disagree on rank 1 ===")
cols = ["idx", "doc_name", "gold_page", "modality"] + ALL
display(res[cols][res[ALL].apply(lambda r: len({hit_at_k(x, 1) for x in r}) > 1, axis=1)])

,ColPali wins @1,text wins @1,ties,sign-test p,dRecall@1,dRecall@1 95% CI,dMRR@10,dMRR@10 95% CI
ocr-bm25,5,2,23,0.4531,0.1,"(-0.067, 0.267)",0.074,"(-0.046, 0.202)"
ocr-dense,16,1,13,0.0003,0.5,"(0.3, 0.7)",0.334,"(0.203, 0.463)"
ocr-hybrid,14,2,14,0.0042,0.4,"(0.167, 0.6)",0.213,"(0.078, 0.342)"


A confidence interval that excludes 0 means the gap is unlikely to be sampling noise.

=== Questions where the arms disagree on rank 1 ===


,idx,doc_name,gold_page,modality,colpali,ocr-bm25,ocr-dense,ocr-hybrid
0,0,2407.01449v6,1,text,7,1.0,3.0,1
1,1,2407.01449v6,2,chart,1,3.0,5.0,5
2,2,2407.01449v6,4,table,1,1.0,2.0,1
5,5,2407.01449v6,9,chart,1,1.0,2.0,2
7,7,2407.01449v6,19,table,1,1.0,2.0,2
8,8,2407.01449v6,21,table,1,4.0,1.0,2
11,11,WorldDevelopmentReport,78,chart,1,1.0,4.0,2
14,14,WorldDevelopmentReport,132,table,1,2.0,4.0,2
15,15,WorldDevelopmentReport,155,chart,1,1.0,2.0,2
17,17,WorldDevelopmentReport,227,infographic,1,1.0,2.0,2


## 8. Generation: same Gemini, same prompt, one page, different retriever

To keep the comparison clean and the Gemini quota small:
- Only the **best text arm by retrieval MRR@10** is carried into generation (picked below from retrieval numbers only).
- When a text arm's top-1 page is the **same page** ColPali retrieved, the identical input would give the same answer, so ColPali's existing answer and verdict are reused (no new call).
- Only questions where the top-1 pages differ call Gemini (generation + judge), using the same system prompt, model and judge prompt as Evaluation 1.
- Verdict rule is unchanged: manual override > LLM judge > strict keyword scorer.

In [13]:
BEST = overall.loc[ARMS, "MRR@10"].idxmax()
print("Best text arm by retrieval MRR@10:", BEST, "->", round(overall.loc[BEST, "MRR@10"], 3))
top1 = {i: TOP10[BEST][i][0] for i in range(len(QA_SET))}
col_top1 = {i: TOP10["colpali"][i][0] for i in range(len(QA_SET))}
need = [i for i in range(len(QA_SET)) if top1[i] != col_top1[i]]
print(f"{len(QA_SET) - len(need)} questions reuse ColPali's answer (same top-1 page); {len(need)} need new Gemini calls")

Best text arm by retrieval MRR@10: ocr-bm25 -> 0.836
22 questions reuse ColPali's answer (same top-1 page); 8 need new Gemini calls


In [14]:
from getpass import getpass
from google import genai
from concurrent.futures import ThreadPoolExecutor

GEMINI_MODEL = EVAL_CFG["generation"]["model"]
SYSTEM_INSTRUCTION = EVAL_CFG["generation"]["system_instruction"]
JUDGE_MODEL = "gemini-3.5-flash-lite"
JUDGE_SYSTEM = (
    "You are a strict grader for a document question-answering evaluation. You are given a question, "
    "a reference answer, a model answer, and the page image the model was shown. Grade the model answer.\n"
    "correct: 1 if the model answer states the reference answer's key fact(s) -- paraphrase, units and "
    "formatting differences are fine; 0 if it is wrong, incomplete on the key fact, or says the page "
    "does not contain the answer.\n"
    "faithful: 1 if every claim in the model answer is supported by the page image shown (an honest "
    "'the page does not contain the answer' counts as faithful); 0 if it asserts something the page "
    "does not support.\n"
    "abstained: 1 if the model answer says the provided page does not contain the answer, else 0.\n"
    "Respond with ONLY a JSON object: {\"correct\": 0|1, \"faithful\": 0|1, \"abstained\": 0|1, "
    "\"reason\": \"<one short sentence>\"}")

_pool, _last, _client = ThreadPoolExecutor(4), [0.0], [None]
def set_key():
    k = getpass("Paste a Gemini API key (blank = stop): ").strip().strip("\"'")
    if not k: return False
    _client[0] = genai.Client(api_key=k); return True

def call(model, parts, system=None, timeout=120):
    for attempt in range(4):
        wait = 13 - (time.time() - _last[0])
        if wait > 0: time.sleep(wait)
        client, kw = _client[0], {"model": model, "input": parts}
        if system: kw["system_instruction"] = system
        try:
            out = _pool.submit(lambda: client.interactions.create(**kw)).result(timeout=timeout)
            _last[0] = time.time(); return out.output_text
        except Exception as e:
            _last[0] = time.time(); t = f"{type(e).__name__}: {e}"; tl = t.lower()
            if any(m in tl for m in ("perday", "per_day", "billing", "free tier", "api key", "permission", "403")):
                print("  key problem:", t[:160])
                if set_key(): continue
                return None
            if any(m in tl for m in ("429", "quota", "resource_exhausted")): time.sleep(min(60, 10 * (attempt + 1))); continue
            if any(m in tl for m in ("503", "unavailable", "overloaded", "500", "timeout", "timed out")): time.sleep(20 * (attempt + 1)); continue
            print("  error:", t[:200]); return None
    return None

def parts_for(text, path):
    with open(path, "rb") as f: b64 = base64.b64encode(f.read()).decode()
    return [{"type": "text", "text": text}, {"type": "image", "data": b64, "mime_type": "image/png"}]

def parse_judge(text):
    if not isinstance(text, str): return None
    m = re.search(r"\{.*\}", re.sub(r"```(?:json)?", "", text), flags=re.S)
    try:
        d = json.loads(m.group(0)); to1 = lambda v: int(str(v).strip().lower() in ("1", "true", "yes"))
        return {"correct": to1(d["correct"]), "faithful": to1(d["faithful"]), "abstained": to1(d.get("abstained", 0)), "reason": str(d.get("reason", ""))}
    except Exception: return None

CK = f"{E2}/eval2_gen_checkpoint_{BEST}.json"
gen = json.load(open(CK)) if os.path.exists(CK) else {}
todo = [i for i in need if str(i) not in gen or gen[str(i)].get("correct") is None]
print(f"{len(need) - len(todo)}/{len(need)} already done")
if todo and not _client[0] and not set_key(): todo = []; print("No key -- skipping generation; re-run later (resumes).")
for i in todo:
    qa = QA_SET[i]; path = PAGE_IMG[top1[i]]
    ans = call(GEMINI_MODEL, parts_for(f"Question: {qa['question']}", path), SYSTEM_INSTRUCTION)
    if ans is None: print("Stopped (no reply / no key). Re-run this cell later; it resumes."); break
    raw = call(JUDGE_MODEL, parts_for(f"Question: {qa['question']}\nReference answer: {qa['gold_answer']}\nModel answer: {ans}\n\nGrade the model answer.", path), JUDGE_SYSTEM)
    pj = parse_judge(raw) or {"correct": None, "faithful": None, "abstained": None, "reason": ""}
    gen[str(i)] = {"idx": i, "page": list(top1[i]), "model_answer": ans, "judge_model": JUDGE_MODEL, **pj}
    json.dump(gen, open(CK, "w"), indent=2)
    print(f"[{i+1:>2}/{len(QA_SET)}] {qa['doc_name']:<24} p.{qa['page']:<4} judge={pj['correct']}")
print("Done." if all(str(i) in gen and gen[str(i)].get("correct") is not None for i in need) else "Some questions are still pending.")

0/8 already done
Paste a Gemini API key (blank = stop): ··········
[ 1/30] 2407.01449v6             p.1    judge=1
[ 2/30] 2407.01449v6             p.2    judge=1
[ 5/30] 2407.01449v6             p.7    judge=0
[ 9/30] 2407.01449v6             p.21   judge=0
[15/30] WorldDevelopmentReport   p.132  judge=1
[26/30] AIFoundations            p.95   judge=0
[28/30] AIFoundations            p.108  judge=1
[30/30] AIFoundations            p.117  judge=0
Done.


In [15]:
# Review the answers a human should look at (abstentions and grader disagreements), as in Evaluation 1.
rows = []
for i in need:
    g = gen.get(str(i));  qa = QA_SET[i]
    if not g: continue
    strict = strict_keyword_score(g["model_answer"], qa["gold_answer"])
    rows.append({**g, "gold_answer": qa["gold_answer"], "question": qa["question"], "strict": strict, "abstained_rule": is_abstention(g["model_answer"])})
for r in rows:
    if r["abstained_rule"] or (r["correct"] is not None and bool(r["correct"]) != bool(r["strict"])):
        print("-" * 90); print(f"idx {r['idx']} | Q: {r['question']}\nGOLD: {r['gold_answer']}\nANS : {r['model_answer'][:350]}")
        print(f"      judge={r['correct']} strict={r['strict']} abstained={r['abstained_rule']} | {r['reason']}")

# Record any verdict you want to override after reading the above: {idx: True/False}
MANUAL_E2 = {}

------------------------------------------------------------------------------------------
idx 4 | Q: According to Table 2, what nDCG@5 score does ColPali achieve on the TabFQuAD (TabF) benchmark?
GOLD: 83.9
ANS : Based on the provided page, Table 2 is not included in the document. Therefore, the requested nDCG@5 score for ColPali on the TabFQuAD benchmark cannot be determined from this page.
      judge=0 strict=False abstained=True | The page does not contain Table 2, so the model correctly abstained, but did not provide the correct answer.
------------------------------------------------------------------------------------------
idx 8 | Q: According to Table 6, what is ColPali's average Recall@1 score across all ViDoRe tasks?
GOLD: 72.7
ANS : Based on the provided document page, Table 6 is not present. Therefore, the image does not contain the information needed to answer your question regarding ColPali's average Recall@1 score across all ViDoRe tasks.
      judge=0 strict=False abs

In [16]:
def final_for(i):
    if top1[i] == col_top1[i]:                                   # identical input -> reuse ColPali's verdict
        r = E1_CSV.loc[E1_CSV.idx == i].iloc[0]
        return bool(r["final_correct"]), "reused_from_eval1", r["judge_faithful"]
    g = gen.get(str(i))
    if i in MANUAL_E2: return bool(MANUAL_E2[i]), "manual", (g or {}).get("faithful")
    if g and g.get("correct") is not None: return bool(g["correct"]), "llm_judge", g.get("faithful")
    return strict_keyword_score(g["model_answer"], QA_SET[i]["gold_answer"]) if g else None, "strict" if g else "pending", None

fin = [final_for(i) for i in range(len(QA_SET))]
res["colpali_correct"] = E1_CSV.sort_values("idx")["final_correct"].astype(bool).tolist()
res[f"{BEST}_correct"] = [f[0] for f in fin]; res[f"{BEST}_verdict_src"] = [f[1] for f in fin]
res[f"{BEST}_faithful"] = [f[2] for f in fin]
done = res[f"{BEST}_correct"].notna()
print(f"Generation finished for {int(done.sum())}/{len(res)} questions\n")
d = res[done].copy(); d[f"{BEST}_correct"] = d[f"{BEST}_correct"].astype(bool)
gtab = pd.DataFrame({"ColPali + MaxSim": [d["colpali_correct"].mean()], BEST: [d[f"{BEST}_correct"].mean()]}, index=["Answer accuracy"])
display(gtab.round(3))
print("By modality:")
display(pd.DataFrame({"n": d.groupby("modality").size(),
                      "ColPali": d.groupby("modality")["colpali_correct"].mean(),
                      BEST: d.groupby("modality")[f"{BEST}_correct"].mean()}).round(3))
a, b = d["colpali_correct"].values, d[f"{BEST}_correct"].values
w, l = int((a & ~b).sum()), int((~a & b).sum())
print(f"Paired end-to-end: ColPali-only correct {w}, text-only correct {l}, sign-test p = {binomtest(w, w + l, 0.5).pvalue if (w + l) else 1.0:.4f}")

Generation finished for 30/30 questions



,ColPali + MaxSim,ocr-bm25
Answer accuracy,0.9,0.833


By modality:


,n,ColPali,ocr-bm25
modality,,,
chart,15,0.867,0.867
diagram,2,1.000,0.500
infographic,1,1.000,1.000
map,2,1.000,1.000
table,7,0.857,0.714
text,3,1.000,1.000


Paired end-to-end: ColPali-only correct 3, text-only correct 1, sign-test p = 0.6250


## 9. Efficiency and storage: where OCR may win

Quoted honestly: ColPali's numbers come from Evaluation 1 (CPU runtime). OCR time is measured here on this runtime. Compare like with like, and say which device each number is from.

In [17]:
txt_kb = np.mean([len(OCR_TEXT[k].encode("utf-8")) for k in PAGES]) / 1024
E1E = E1_SUM["efficiency"]
eff = pd.DataFrame([
    {"Arm": "ColPali + MaxSim", "Indexing / page": "not measured (paper: 0.39 s on its GPU)",
     "Storage / page": f"{E1E['storage']['patch_matrix_kb_per_page']:.0f} KB (patch matrix)",
     "Query: encode": f"{E1E['query_latency_ms']['embedding']['mean_ms']/1000:.1f} s ({E1E['device']})",
     "Query: search": f"{E1E['query_latency_ms']['maxsim_exhaustive']['mean_ms']:.0f} ms (exhaustive MaxSim)"},
    {"Arm": "OCR + BM25", "Indexing / page": f"{np.mean(ocr_secs):.1f} s OCR (CPU)",
     "Storage / page": f"{txt_kb:.1f} KB (text)", "Query: encode": "none",
     "Query: search": f"{np.mean(LAT['ocr-bm25']):.0f} ms"},
    {"Arm": "OCR + dense", "Indexing / page": f"{np.mean(ocr_secs):.1f} s OCR + embedding",
     "Storage / page": f"{txt_kb:.1f} KB text + {np.load(f'{E2}/dense_ocr.npy').nbytes/len(PAGES)/1024:.1f} KB vectors",
     "Query: encode+search": f"{np.mean(LAT['ocr-dense']):.0f} ms ({DEVICE})"},
])
display(eff.fillna(""))

,Arm,Indexing / page,Storage / page,Query: encode,Query: search,Query: encode+search
0,ColPali + MaxSim,not measured (paper: 0.39 s on its GPU),258 KB (patch matrix),17.2 s (cpu),344 ms (exhaustive MaxSim),
1,OCR + BM25,20.1 s OCR (CPU),3.4 KB (text),none,6 ms,
2,OCR + dense,20.1 s OCR + embedding,3.4 KB text + 6.5 KB vectors,,,130 ms (cpu)


## 10. Save everything for the report and presentation

In [18]:
def _v(p):
    try: return _im.version(p)
    except Exception: return None
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (isinstance(o, float) and math.isnan(o)) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return o

res.to_csv(f"{E2}/eval2_per_question.csv", index=False)
summary = {
    "qa_hash": QA_HASH, "best_text_arm": BEST, "arms": ALL,
    "retrieval_overall": overall.to_dict("index"),
    "recall1_by_modality": by_mod.to_dict("index"),
    "paired_vs_colpali": PAIRED.astype(str).to_dict("index"),
    "generation": {"n_finished": int(done.sum()), "colpali_accuracy": float(d["colpali_correct"].mean()),
                   f"{BEST}_accuracy": float(d[f"{BEST}_correct"].mean()), "reused_from_eval1": len(QA_SET) - len(need),
                   "new_gemini_calls": len(need)},
    "efficiency": {"ocr_s_per_page_mean": float(np.mean(ocr_secs)), "text_kb_per_page": float(txt_kb),
                   "bm25_ms": float(np.mean(LAT["ocr-bm25"])), "dense_ms": float(np.mean(LAT["ocr-dense"]))},
    "setup": {"ocr_engine": "tesseract " + str(pytesseract.get_tesseract_version()), "dense_model": "BAAI/bge-small-en-v1.5",
              "chunk_words": 180, "chunk_stride": 120, "rrf_k": 60, "native_text_arm_included": bool(PDF_PATHS),
              "python": platform.python_version(),
              "versions": {p: _v(p) for p in ("pytesseract", "rank-bm25", "sentence-transformers", "torch", "numpy", "pandas", "scipy")}},
}
json.dump(_clean(summary), open(f"{E2}/eval2_summary.json", "w"), indent=2)
with open(f"{E2}/eval2_report_tables.md", "w") as f:
    f.write("## Retrieval (overall)\n\n" + overall.round(3).to_markdown() + "\n\n## Recall@1 by modality\n\n" + by_mod.round(3).to_markdown()
            + "\n\n## Paired comparison vs ColPali\n\n" + PAIRED.astype(str).to_markdown() + "\n\n## Efficiency\n\n" + eff.fillna("").to_markdown() + "\n")
print("Saved to", E2)
print(" - eval2_per_question.csv  - eval2_summary.json  - eval2_report_tables.md")

Saved to /content/drive/MyDrive/Visual-Doc/eval_results/eval2
 - eval2_per_question.csv  - eval2_summary.json  - eval2_report_tables.md


## Reading the results honestly

- If ColPali wins on chart / table / map / diagram questions and the text arms tie or win on text questions, that is the finding, and it is exactly what the visual-retrieval argument predicts. Report it as such.
- The benchmark is chart-heavy (15 of 30), and most questions name a figure or table, so captions such as "Figure 2" give keyword search a head start. State both facts in the report.
- 30 questions is small: lean on the paired intervals in Section 7, not on the point estimates alone.
- Evaluation 3 (a general assistant such as ChatGPT with PDF upload) stays a separate, uncontrolled positioning check.